# Assignment 1 - LLM Evaluation: Product Description Generation

**Course:** AI Performance Engineering  
**Due Date:** April 5, 2026

This notebook contains the complete solution for Assignment 1, covering:
1. Rubric definition
2. Description generation
3. Manual evaluation
4. Improvement cycle
5. Judge model creation
6. Judge analysis and comparison

---
## Task 1: Define Your Rubric (15 points)

Before generating or evaluating anything, we need a clear, repeatable scoring framework.

### 1.1 Criterion Definitions

For each criterion, we define explicit standards for **good**, **ok**, and **bad** ratings. These are stored in a structured dictionary for programmatic evaluation.

### 1.2 Criterion Thresholds Dictionary

In [ ]:
# Criterion thresholds for automated evaluation
CRITERION_THRESHOLDS = {
    "fluency": {
        "check": "manual",
        "good": {
            "description": "Natural, smooth sentences with varied structure. Easy to read aloud. No awkward phrasing or repetition.",
        },
        "ok": {
            "description": "Mostly natural but with minor awkwardness (e.g., one slightly repetitive phrase or choppy transition).",
        },
        "bad": {
            "description": "Multiple awkward phrases, unnatural word order, or repetitive structure that disrupts readability.",
        },
    },
    "grammar": {
        "check": "manual",
        "good": {
            "description": "Zero spelling or punctuation errors. Proper sentence structure throughout.",
        },
        "ok": {
            "description": "One minor error (e.g., missing comma, minor typo) that doesn't affect comprehension.",
        },
        "bad": {
            "description": "Multiple errors or one major error (e.g., subject-verb disagreement, misspelled product name).",
        },
    },
    "tone": {
        "check": "manual",
        "good": {
            "description": "Consistently friendly, credible sales voice. Enthusiastic without being pushy. Professional language appropriate for e-commerce.",
        },
        "ok": {
            "description": "Generally appropriate tone but with one instance of overly casual language, excessive hype, or slightly flat delivery.",
        },
        "bad": {
            "description": "Inappropriate tone (too formal/technical, too casual, or overly aggressive sales language). Multiple tone inconsistencies.",
        },
    },
    "length": {
        "check": "automatic",
        "good": {
            "description": "50-90 words (inclusive)",
            "ranges": [(50, 90)],
        },
        "ok": {
            "description": "40-49 words OR 91-110 words",
            "ranges": [(40, 49), (91, 110)],
        },
        "bad": {
            "description": "Fewer than 40 words OR more than 110 words",
            "ranges": [(0, 39), (110, float("inf"))],
        },
    },
    "grounding": {
        "check": "manual",
        "good": {
            "description": "All information comes directly from provided data (name, attributes, material, warranty). No fabricated features or specifications.",
        },
        "ok": {
            "description": "Minor embellishment that's reasonable inference (e.g., 'sleek design' when material is 'aluminum') but no false claims.",
        },
        "bad": {
            "description": "Contains fabricated information, incorrect specifications, or claims not supported by the provided data.",
        },
    },
    "latency": {
        "check": "automatic",
        "good": {
            "description": "≤ 5000ms (5 seconds)",
            "ranges": [(0, 5000)],
        },
        "ok": {
            "description": "5001-10000ms (5-10 seconds)",
            "ranges": [(5001, 10000)],
        },
        "bad": {
            "description": "> 10000ms (10+ seconds)",
            "ranges": [(10001, float("inf"))],
        },
    },
    "cost": {
        "check": "automatic",
        "good": {
            "description": "≤ $0.01 per description",
            "ranges": [(0, 0.01)],
        },
        "ok": {
            "description": "$0.011-$0.05 per description",
            "ranges": [(0.011, 0.05)],
        },
        "bad": {
            "description": "> $0.05 per description",
            "ranges": [(0.051, float("inf"))],
        },
    },
}


def evaluate_criterion(criterion_name: str, value: float) -> str:
    """
    Generic evaluation function for any criterion with numeric ranges.

    Args:
        criterion_name: Name of the criterion (e.g., 'length', 'latency', 'cost')
        value: Numeric value to evaluate

    Returns:
        'good', 'ok', or 'bad'
    """
    thresholds = CRITERION_THRESHOLDS[criterion_name]

    for rating in ["good", "ok", "bad"]:
        for min_val, max_val in thresholds[rating]["ranges"]:
            if min_val <= value <= max_val:
                return rating

    return "bad"


def evaluate_length(word_count: int) -> str:
    """Evaluate length criterion based on word count using CRITERION_THRESHOLDS."""
    return evaluate_criterion("length", word_count)


def evaluate_latency(latency_ms: float) -> str:
    """Evaluate latency criterion based on milliseconds using CRITERION_THRESHOLDS."""
    return evaluate_criterion("latency", latency_ms)


def evaluate_cost(cost_usd: float) -> str:
    """Evaluate cost criterion based on USD amount using CRITERION_THRESHOLDS."""
    return evaluate_criterion("cost", cost_usd)


def calculate_pass_fail(ratings: dict) -> str:
    """
    Calculate pass/fail based on ratings.

    Args:
        ratings: dict with keys ['fluency', 'grammar', 'tone', 'length', 'grounding', 'latency', 'cost']
                 values: 'good', 'ok', or 'bad'

    Returns:
        'pass' or 'fail'

    Rules:
        - Automatic failure if grounding, grammar, or length is "bad"
        - Pass requires: ≥4 good, ≤1 bad, and ≥5 acceptable (good or ok)
    """
    # Go/no-go rules - automatic failure conditions
    if ratings["grounding"] == "bad":
        return "fail"
    if ratings["grammar"] == "bad":
        return "fail"
    if ratings["length"] == "bad":
        return "fail"

    # Cumulative pass bar - count ratings by type
    good_count = sum(1 for v in ratings.values() if v == "good")
    ok_count = sum(1 for v in ratings.values() if v == "ok")
    bad_count = sum(1 for v in ratings.values() if v == "bad")

    # Calculate combined acceptable ratings
    good_or_ok_count = good_count + ok_count

    # Must have: ≥4 good, ≤1 bad, and ≥5 acceptable (good or ok)
    if (good_count >= 4) and (bad_count <= 1) and (good_or_ok_count >= 5):
        return "pass"
    else:
        return "fail"

---
## Task 2: Generate Descriptions for Every Product (20 points)

Generate product descriptions using a language model from Nebius Token Factory.

In [20]:
# Import required libraries
import os
import time

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables
load_dotenv()

# Constants
NEBIUS_API_BASE_URL = "https://api.tokenfactory.nebius.com/v1/"
# split criteria into quality and objective
QUALITY_CRITERIA = ["fluency", "grammar", "tone", "length", "grounding"]
OBJECTIVE_CRITERIA = ["latency", "cost"]
EVALUATION_CRITERIA = [
    *QUALITY_CRITERIA,
    *OBJECTIVE_CRITERIA,
]

# Initialize OpenAI client
client = OpenAI(base_url=NEBIUS_API_BASE_URL, api_key=os.environ.get("NEBIUS_API_KEY"))

In [21]:
EVALUATION_CRITERIA

['fluency', 'grammar', 'tone', 'length', 'grounding', 'latency', 'cost']

In [22]:
# Load the product dataset
df = pd.read_csv("Assignment_01_product_dataset.csv")
print(f"Loaded {len(df)} products")
df.head()

Loaded 50 products


,product_name,Product_attribute_list,material,warranty
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1-year limited warranty
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S-Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1-year limited warranty
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1-year limited warranty
3,Sony WH-1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1-year limited warranty
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1-year limited warranty


### 2.1 System Prompt

Design a prompt that instructs the model to generate persuasive 50-90 word product descriptions.

In [23]:
SYSTEM_PROMPT = """
You are an expert e-commerce copywriter. Your task is to write persuasive product descriptions for online shoppers.

Requirements:
- Length: Exactly 50-90 words
- Tone: Friendly, credible, and enthusiastic (but not pushy)
- Content: Use ONLY the provided product information - do not fabricate features
- Style: Natural, easy-to-read sentences with varied structure
- Grammar: Perfect spelling and punctuation

Focus on benefits and appeal to the target customer. Make them want to buy!

OUTPUT: Provide only the product description text. Do not include any preamble, explanation, or additional commentary.
""".strip()


def create_user_prompt(
    product_name: str, attributes: str, material: str, warranty: str
) -> str:
    return f"""Product Name: {product_name}
Attributes: {attributes}
Material: {material}
Warranty: {warranty}

Write a persuasive product description (50-90 words)."""

### 2.2 Model Selection

Choose one model from Nebius Token Factory:
- Gemma-2-9b-it
- Meta-Llama-3.1-8B-Instruct

In [24]:
# Choosing the model
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"  # or "google/gemma-2-9b-it"


def generate_description(
    product_name: str, attributes: str, material: str, warranty: str
) -> dict:
    """
    Generate a product description and collect metrics.

    Returns:
        dict with keys: generated_description, latency_ms, input_tokens, output_tokens
    """
    user_prompt = create_user_prompt(product_name, attributes, material, warranty)

    start_time = time.time()

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        max_completion_tokens=150,
    )

    end_time = time.time()
    latency_ms = int((end_time - start_time) * 1000)

    return {
        "generated_description": response.choices[0].message.content.strip(),
        "latency_ms": latency_ms,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
    }

### 2.3 Generate Descriptions for All Products

In [12]:
# Run generation for all products
results = []

for idx, row in df.iterrows():
    print(f"Processing {idx + 1}/{len(df)}: {row['product_name']}")

    result = generate_description(
        product_name=row["product_name"],
        attributes=row["Product_attribute_list"],
        material=row["material"],
        warranty=row["warranty"],
    )

    # Combine original data with generated results
    results.append({**row.to_dict(), **result})

    # Small delay to avoid rate limiting
    time.sleep(0.5)

print("\nGeneration complete!")

Processing 1/50: Apple iPhone 15 Pro
Processing 2/50: Samsung Galaxy S24 Ultra
Processing 3/50: Google Pixel 8 Pro
Processing 4/50: Sony WH-1000XM5 Headphones
Processing 5/50: Bose QuietComfort Ultra Earbuds
Processing 6/50: Amazon Echo Dot (5th Gen)
Processing 7/50: Dell XPS 13 9310 Laptop
Processing 8/50: Apple MacBook Air 13″ (M3)
Processing 9/50: Microsoft Surface Pro 10
Processing 10/50: Garmin Forerunner 255
Processing 11/50: Fitbit Charge 6
Processing 12/50: GoPro HERO12 Black
Processing 13/50: DJI Mini 4 Pro Drone
Processing 14/50: Nintendo Switch OLED
Processing 15/50: PlayStation 5 Slim
Processing 16/50: Xbox Series X
Processing 17/50: Instant Pot Duo 6-Quart
Processing 18/50: Keurig K-Supreme Plus Smart
Processing 19/50: Vitamix 5200 Blender
Processing 20/50: Dyson V15 Detect Vacuum
Processing 21/50: iRobot Roomba j7+
Processing 22/50: Yeti Rambler 20 oz Tumbler
Processing 23/50: Stanley Quencher H2.0 40 oz
Processing 24/50: Hydro Flask 32 oz Wide Mouth
Processing 25/50: Con

### 2.4 Create DataFrame and Save to Excel

In [16]:
# Create results DataFrame
results_df = pd.DataFrame(results)

# Add blank columns for evaluation criteria
for criterion in EVALUATION_CRITERIA:
    results_df[criterion] = ""

results_df["final_score"] = ""

# Save to Excel
results_df.to_excel("assignment_01.xlsx", index=False)
print("Saved results to assignment_01.xlsx")

# Display summary
print(f"\nGenerated {len(results_df)} descriptions")
print(f"Average latency: {results_df['latency_ms'].mean():.0f}ms")
print(f"Average input tokens: {results_df['input_tokens'].mean():.0f}")
print(f"Average output tokens: {results_df['output_tokens'].mean():.0f}")

results_df.head()

Saved results to assignment_01.xlsx

Generated 50 descriptions
Average latency: 8321ms
Average input tokens: 197
Average output tokens: 105


,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,tone,length,grounding,latency,cost,final_score
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1-year limited warranty,"""Upgrade your mobile experience with the Apple...",11975,200,108,,,,,,,,
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S-Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1-year limited warranty,"""Unlock your creativity with the Samsung Galax...",9583,203,112,,,,,,,,
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1-year limited warranty,"""Unlock the power of unparalleled photography ...",14984,202,109,,,,,,,,
3,Sony WH-1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1-year limited warranty,Escape the noise with the Sony WH-1000XM5 Head...,9036,201,116,,,,,,,,
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1-year limited warranty,Immerse yourself in the perfect blend of sound...,7361,194,115,,,,,,,,


---
## Task 3: Manual (Human) Evaluation (10 points)

Manually evaluate 10-15 products using the rubric defined in Task 1.

### 3.1 Add Cost Column

Calculate the cost per description based on token usage and model pricing.

In [25]:
# Fetch pricing dynamically from Nebius API
def get_model_pricing(model_name: str) -> tuple[float, float]:
    """
    Fetch pricing for a specific model from Nebius Token Factory API.

    Args:
        model_name: The model identifier (e.g., "meta-llama/Meta-Llama-3.1-8B-Instruct")

    Returns:
        Tuple of (input_price_per_1k_tokens, output_price_per_1k_tokens) in USD
    """
    try:
        # List all models with verbose=true to get pricing information
        # Note: The OpenAI client doesn't support query params directly,
        # so we need to make a raw HTTP request
        import requests

        response = requests.get(
            f"{NEBIUS_API_BASE_URL}models?verbose=true",
            headers={"Authorization": f"Bearer {os.environ.get('NEBIUS_API_KEY')}"},
        )
        response.raise_for_status()
        models_data = response.json()

        # Find the specific model
        for model in models_data.get("data", []):
            if model.get("id") == model_name:
                pricing = model.get("pricing", {})

                # Pricing values are strings representing price per token
                # Convert to float and then to per 1K tokens
                input_price_per_token = float(pricing.get("prompt", "0"))
                output_price_per_token = float(pricing.get("completion", "0"))

                input_price_per_1k = input_price_per_token * 1000
                output_price_per_1k = output_price_per_token * 1000

                print(f"✓ Fetched pricing for {model_name}:")
                print(f"  Input:  ${input_price_per_1k:.6f} per 1K tokens")
                print(f"  Output: ${output_price_per_1k:.6f} per 1K tokens")
                return input_price_per_1k, output_price_per_1k

        # Model not found
        raise ValueError(f"Model not found: {model_name}")

    except Exception as e:
        print(f"⚠ Warning: Could not fetch pricing from API: {e}")
        print("Using fallback pricing values for meta-llama/Meta-Llama-3.1-8B-Instruct")
        # Fallback: $0.02/1M input, $0.06/1M output = $0.00002/1K, $0.00006/1K
        return 0.00002, 0.00006


# Get pricing for the chosen model
PRICE_PER_1K_INPUT_TOKENS, PRICE_PER_1K_OUTPUT_TOKENS = get_model_pricing(MODEL_NAME)

# Load the Excel file
results_df = pd.read_excel("assignment_01.xlsx")

# Calculate cost
results_df["cost"] = (results_df["input_tokens"] / 1000 * PRICE_PER_1K_INPUT_TOKENS) + (
    results_df["output_tokens"] / 1000 * PRICE_PER_1K_OUTPUT_TOKENS
)

print(f"\nAverage cost per description: ${results_df['cost'].mean():.6f}")
print(f"Total cost for {len(results_df)} descriptions: ${results_df['cost'].sum():.6f}")

# Save the results
results_df.to_excel("assignment_01.xlsx", index=False)

✓ Fetched pricing for meta-llama/Meta-Llama-3.1-8B-Instruct:
  Input:  $0.000020 per 1K tokens
  Output: $0.000060 per 1K tokens

Average cost per description: $0.000010
Total cost for 50 descriptions: $0.000513


### 3.2 Manual Evaluation Instructions

**TODO: Manually evaluate 10-15 products**

1. Open `assignment_01.xlsx` in Excel
2. Select 10-15 diverse products (mix of different categories)
3. For each selected product, rate each criterion (fluency, grammar, tone, length, grounding, latency, cost) as:
   - `good`
   - `ok`
   - `bad`
4. Use the rubric definitions from Task 1
5. Calculate `final_score` (pass/fail) using the formula from Task 1
6. Save the Excel file

After completing manual evaluation, run the cell below to load and analyze your scores.

In [42]:
def evaluate_and_save_automatic_criteria(
    excel_path: str, n_rows: int = None, save: bool = True
) -> pd.DataFrame:
    """
    Evaluate automatic criteria (length, latency, cost) and optionally save back to Excel.

    Args:
        excel_path: Path to the Excel file
        n_rows: Number of rows to evaluate (None = all rows)
        save: Whether to save results back to the Excel file

    Returns:
        DataFrame with automatic evaluations
    """
    # Read Excel file
    df = pd.read_excel(excel_path)

    # Mapping from criterion name to DataFrame column name
    criterion_to_column = {
        "length": "generated_description",  # Need to count words
        "latency": "latency_ms",
        "cost": "cost",
    }

    # Helper function to count words
    def count_words(text):
        return len(str(text).split())

    # Limit to first n rows if specified
    if n_rows is not None:
        eval_mask = df.index < n_rows
    else:
        eval_mask = df.index >= 0  # All rows

    # Calculate word counts for rows to evaluate
    word_counts = df.loc[eval_mask, "generated_description"].apply(count_words)

    # Evaluate each automatic criterion and update existing columns
    for criterion_name in CRITERION_THRESHOLDS.keys():
        if CRITERION_THRESHOLDS[criterion_name]["check"] == "automatic":
            if criterion_name == "length":
                # For length, use word_count
                df.loc[eval_mask, criterion_name] = word_counts.apply(
                    lambda x: evaluate_criterion("length", x)
                )
            else:
                # For latency and cost, use the mapped column
                col_name = criterion_to_column[criterion_name]
                df.loc[eval_mask, criterion_name] = df.loc[eval_mask, col_name].apply(
                    lambda x: evaluate_criterion(criterion_name, x)
                )

    # Save back to Excel file if requested
    if save:
        df.to_excel(excel_path, index=False)
        rows_evaluated = n_rows if n_rows is not None else len(df)
        print(f"✓ Automatically evaluated {rows_evaluated} rows")
        print(f"✓ Updated columns: {criterion_to_column.keys()}")
        print(f"✓ Saved to {excel_path}")

    return df


# Evaluate first 15 rows and save
test_df = evaluate_and_save_automatic_criteria("assignment_01.xlsx", n_rows=15)

# Display relevant columns for verification
display_cols = [
    "product_name",
    "length",
    "latency_ms",
    "latency",
    "cost",
]
print("\nFirst 15 rows with automatic evaluations:")
test_df[display_cols].head(15)

TypeError: Invalid value '<StringArray>
['good', 'good', 'good', 'good', 'good', 'good', 'good', 'good', 'good',
 'good', 'good', 'good', 'good', 'good', 'good']
Length: 15, dtype: str' for dtype 'float64'

In [ ]:
# Load manually evaluated data
evaluated_df = pd.read_excel("assignment_01.xlsx")

# Filter rows with manual evaluation (non-empty fluency column)
manual_eval = evaluated_df[evaluated_df["fluency"] != ""].copy()

print(f"Manually evaluated {len(manual_eval)} products\n")

# Analyze baseline performance
print("Baseline Analysis - Criterion Performance:")
print("=" * 50)

for criterion in EVALUATION_CRITERIA:
    if criterion in manual_eval.columns:
        value_counts = manual_eval[criterion].value_counts()
        print(f"\n{criterion.upper()}:")

        for rating in ["good", "ok", "bad"]:
            count = value_counts.get(rating, 0)
            pct = (count / len(manual_eval) * 100) if len(manual_eval) > 0 else 0
            print(f"  {rating}: {count} ({pct:.1f}%)")

# Pass/fail summary
if "final_score" in manual_eval.columns:
    final_counts = manual_eval["final_score"].value_counts()
    print(f"\n{'=' * 50}")
    print("FINAL SCORES:")
    print(f"  Pass: {final_counts.get('pass', 0)}")
    print(f"  Fail: {final_counts.get('fail', 0)}")
    pass_rate = (
        (final_counts.get("pass", 0) / len(manual_eval) * 100)
        if len(manual_eval) > 0
        else 0
    )
    print(f"  Pass rate: {pass_rate:.1f}%")

### 3.3 Baseline Analysis

**TODO: Document your findings**

Based on the manual evaluation:

1. **Best performing criteria:**
   - [Write your analysis here]

2. **Worst performing criteria:**
   - [Write your analysis here]

3. **Common failure patterns:**
   - [Write your analysis here]

4. **Strategy for improvement (Task 4):**
   - [Write your improvement strategy here]

---
## Task 4: Improvement Cycle (15 points)

Iterate to achieve better results based on Task 3 baseline analysis.

### Experiment Template

For each experiment, document:
1. **What you changed**
2. **Why you expected it to help**
3. **New evaluation scores**

Keep code for successful experiments. Document failed experiments but code is optional.

### Experiment 1: [Name your experiment]

**What changed:**
- [Describe the change]

**Why expected to help:**
- [Explain your hypothesis]

**Results:**
- [Document the outcome]

In [ ]:
# TODO: Experiment 1 code
# Example: Modified prompt, different temperature, etc.

### Experiment 2: [Name your experiment]

**What changed:**
- [Describe the change]

**Why expected to help:**
- [Explain your hypothesis]

**Results:**
- [Document the outcome]

In [ ]:
# TODO: Experiment 2 code

### Experiment 3: [Name your experiment]

**What changed:**
- [Describe the change]

**Why expected to help:**
- [Explain your hypothesis]

**Results:**
- [Document the outcome]

In [ ]:
# TODO: Experiment 3 code

---
## Task 5: Create a Judge Model (20 points)

Build an automated LLM judge that grades descriptions using the Task 1 rubric.

### 5.1 Judge Model Selection

Start with the model you **did not** use in Task 2. If it struggles, switch to a larger model.

In [ ]:
# TODO: Choose judge model (the one NOT used in Task 2)
JUDGE_MODEL_NAME = "google/gemma-2-9b-it"  # or another model if needed

print(f"Judge model: {JUDGE_MODEL_NAME}")
print(f"Generator model was: {MODEL_NAME}")

### 5.2 Pydantic Schema for Structured Output

Define the output schema. Note: **explanation comes before verdict** (important for chain-of-thought reasoning).

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class CriterionEvaluation(BaseModel):
    explanation: str = Field(description="Reasoning for the verdict")
    verdict: Literal["good", "ok", "bad"] = Field(
        description="Rating: good, ok, or bad"
    )


class DescriptionEvaluation(BaseModel):
    fluency: CriterionEvaluation
    grammar: CriterionEvaluation
    tone: CriterionEvaluation
    length: CriterionEvaluation
    grounding: CriterionEvaluation


# Display schema
print("Judge output schema:")
print(DescriptionEvaluation.model_json_schema())

**Why explanation before verdict?**

[TODO: Explain why this ordering matters for LLM reasoning]

### 5.3 Judge Prompt

Write a prompt that embeds the Task 1 rubric and provides necessary context for evaluation.

In [ ]:
JUDGE_SYSTEM_PROMPT = """
You are an expert evaluator of product descriptions. Your task is to rate product descriptions according to specific criteria.

For each criterion, provide:
1. An explanation of your reasoning
2. A verdict: 'good', 'ok', or 'bad'

EVALUATION CRITERIA:

FLUENCY:
- good: Natural, smooth sentences with varied structure. Easy to read aloud. No awkward phrasing or repetition.
- ok: Mostly natural but with minor awkwardness (e.g., one slightly repetitive phrase or choppy transition).
- bad: Multiple awkward phrases, unnatural word order, or repetitive structure that disrupts readability.

GRAMMAR:
- good: Zero spelling or punctuation errors. Proper sentence structure throughout.
- ok: One minor error (e.g., missing comma, minor typo) that doesn't affect comprehension.
- bad: Multiple errors or one major error (e.g., subject-verb disagreement, misspelled product name).

TONE:
- good: Consistently friendly, credible sales voice. Enthusiastic without being pushy. Professional language appropriate for e-commerce.
- ok: Generally appropriate tone but with one instance of overly casual language, excessive hype, or slightly flat delivery.
- bad: Inappropriate tone (too formal/technical, too casual, or overly aggressive sales language). Multiple tone inconsistencies.

LENGTH:
- good: 50-90 words (inclusive)
- ok: 40-49 words OR 91-110 words
- bad: Fewer than 40 words OR more than 110 words

GROUNDING:
- good: All information comes directly from provided product data. No fabricated features or specifications.
- ok: Minor embellishment that's reasonable inference but no false claims.
- bad: Contains fabricated information, incorrect specifications, or claims not supported by the provided data.

Be objective and consistent in your evaluations.
""".strip()


def create_judge_prompt(
    description: str, product_name: str, attributes: str, material: str, warranty: str
) -> str:
    return f"""PRODUCT INFORMATION:
Name: {product_name}
Attributes: {attributes}
Material: {material}
Warranty: {warranty}

GENERATED DESCRIPTION:
{description}

Evaluate this description according to the criteria (fluency, grammar, tone, length, grounding)."""


print("Judge prompt created")

### 5.4 Judge Implementation

In [ ]:
def judge_description(
    description: str, product_name: str, attributes: str, material: str, warranty: str
) -> DescriptionEvaluation:
    """
    Use the judge model to evaluate a product description.

    Returns:
        DescriptionEvaluation object with ratings for each criterion
    """
    user_prompt = create_judge_prompt(
        description, product_name, attributes, material, warranty
    )

    response = client.chat.completions.create(
        model=JUDGE_MODEL_NAME,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,  # Lower temperature for more consistent evaluation
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "description_evaluation",
                "schema": DescriptionEvaluation.model_json_schema(),
            },
        },
    )

    content = response.choices[0].message.content

    # Parse into Pydantic model
    return DescriptionEvaluation.model_validate_json(content)


print("Judge function ready")

---
## Task 6: Run and Analyze the Judge (20 points)

Run the judge model and compare its evaluations to human ratings.

### 6.1 Sanity Check (5 products)

In [ ]:
# TODO: Run judge on 5 products for sanity check
results_df = pd.read_excel("assignment_01.xlsx")

# Select 5 products (can be random or specific)
sanity_check_indices = [0, 10, 20, 30, 40]  # Adjust as needed

print("Sanity Check - Judge Evaluations:")
print("=" * 80)

for idx in sanity_check_indices:
    row = results_df.iloc[idx]

    print(f"\nProduct: {row['product_name']}")
    print(f"Description: {row['generated_description'][:100]}...")

    evaluation = judge_description(
        description=row["generated_description"],
        product_name=row["product_name"],
        attributes=row["Product_attribute_list"],
        material=row["material"],
        warranty=row["warranty"],
    )

    print("\nJudge Ratings:")
    for criterion in ["fluency", "grammar", "tone", "length", "grounding"]:
        eval_obj = getattr(evaluation, criterion)
        print(f"  {criterion}: {eval_obj.verdict}")
        print(f"    → {eval_obj.explanation}")

    time.sleep(1)  # Rate limiting

print("\n" + "=" * 80)
print("Review the explanations and verdicts above.")
print("Does the judge apply your rubric correctly? Adjust prompt if needed.")

**Sanity Check Analysis:**

[TODO: Review the judge outputs above. Do they make sense? Does the judge apply your rubric correctly? Document any issues and prompt adjustments needed.]

### 6.2 Full Run - Judge All Products

In [ ]:
# TODO: Run judge on all products
results_df = pd.read_excel("assignment_01.xlsx")

# Add judge columns
judge_columns = [
    "judge_fluency",
    "judge_fluency_explanation",
    "judge_grammar",
    "judge_grammar_explanation",
    "judge_tone",
    "judge_tone_explanation",
    "judge_length",
    "judge_length_explanation",
    "judge_grounding",
    "judge_grounding_explanation",
    "judge_final_score",
]

for col in judge_columns:
    if col not in results_df.columns:
        results_df[col] = ""

print(f"Running judge on {len(results_df)} products...")

for idx, row in results_df.iterrows():
    print(f"Judging {idx + 1}/{len(results_df)}: {row['product_name']}")

    evaluation = judge_description(
        description=row["generated_description"],
        product_name=row["product_name"],
        attributes=row["Product_attribute_list"],
        material=row["material"],
        warranty=row["warranty"],
    )

    # Store judge ratings
    for criterion in QUALITY_CRITERIA:
        eval_obj = getattr(evaluation, criterion)
        results_df.at[idx, f"judge_{criterion}"] = eval_obj.verdict
        results_df.at[idx, f"judge_{criterion}_explanation"] = eval_obj.explanation

    # Calculate judge final score using Task 1 formula
    judge_ratings = {
        "fluency": results_df.at[idx, "judge_fluency"],
        "grammar": results_df.at[idx, "judge_grammar"],
        "tone": results_df.at[idx, "judge_tone"],
        "length": results_df.at[idx, "judge_length"],
        "grounding": results_df.at[idx, "judge_grounding"],
        "latency": results_df.at[idx, "latency"],  # From Task 2
        "cost": results_df.at[idx, "cost"],  # From Task 3
    }

    # Apply pass/fail formula from Task 1
    # TODO: Implement calculate_pass_fail function or inline logic
    results_df.at[idx, "judge_final_score"] = "pass"  # Placeholder

    time.sleep(1)  # Rate limiting

# Save updated results
results_df.to_excel("assignment_01.xlsx", index=False)
print("\nJudge evaluation complete! Results saved to assignment_01.xlsx")

### 6.3 Compare Judge vs Human Evaluation

In [ ]:
# Load results with both human and judge evaluations
results_df = pd.read_excel("assignment_01.xlsx")

# Filter rows with human evaluation
compared = results_df[results_df["fluency"] != ""].copy()

print(f"Comparing judge vs human on {len(compared)} products\n")
print("Agreement Rates by Criterion:")
print("=" * 50)


for criterion in QUALITY_CRITERIA:
    human_col = criterion
    judge_col = f"judge_{criterion}"

    if human_col in compared.columns and judge_col in compared.columns:
        agreements = (compared[human_col] == compared[judge_col]).sum()
        total = len(compared)
        agreement_rate = (agreements / total * 100) if total > 0 else 0

        print(f"\n{criterion.upper()}:")
        print(f"  Agreement: {agreements}/{total} ({agreement_rate:.1f}%)")

        # Show disagreements
        disagreements = compared[compared[human_col] != compared[judge_col]]
        if len(disagreements) > 0:
            print("  Disagreements:")
            for _, row in disagreements.iterrows():
                print(
                    f"    - {row['product_name'][:40]}: Human={row[human_col]}, Judge={row[judge_col]}"
                )

# Overall agreement
print(f"\n{'=' * 50}")
print("Overall Analysis:")
# TODO: Calculate overall agreement and analyze patterns

**Analysis of Judge vs Human Agreement:**

[TODO: Document your findings]

1. **Where do they agree most?**
   - 

2. **Where do they diverge?**
   - 

3. **Why might these differences occur?**
   - 

### 6.4 Criterion-by-Criterion Judging

Run the judge separately for each criterion (one API call per criterion per product).

In [ ]:
# TODO: Implement single-criterion judge
def judge_single_criterion(
    description: str,
    product_name: str,
    attributes: str,
    material: str,
    warranty: str,
    criterion: str,
) -> CriterionEvaluation:
    """
    Judge a single criterion in isolation.

    Args:
        criterion: One of 'fluency', 'grammar', 'tone', 'length', 'grounding'
    """
    # Create criterion-specific prompt
    criterion_prompt = f"""PRODUCT INFORMATION:
Name: {product_name}
Attributes: {attributes}
Material: {material}
Warranty: {warranty}

GENERATED DESCRIPTION:
{description}

Evaluate ONLY the {criterion.upper()} of this description according to the rubric."""

    # TODO: Implement API call for single criterion
    # Similar to judge_description but returns only CriterionEvaluation
    pass


# Run criterion-by-criterion evaluation on a subset
# TODO: Implement and compare results

**Criterion-by-Criterion Analysis:**

[TODO: Answer these questions]

1. **Did isolating criteria change the results?**
   - 

2. **Why might this approach lead to different outcomes?**
   - 

3. **Did agreement with human scores improve?**
   - 

### 6.5 Final Analysis and Reflection

#### Question 1: Trade-offs between human evaluation and LLM-as-a-judge

Consider: cost, scale, consistency, accuracy

[TODO: Write your analysis here]

**Human Evaluation:**
- Pros:
  - 
- Cons:
  - 

**LLM-as-a-Judge:**
- Pros:
  - 
- Cons:
  - 

#### Question 2: Recommendation for production system

For a production system generating thousands of descriptions daily:

[TODO: Write your recommendation here]

**Recommended approach:**
- 

**Justification:**
- 

**Implementation considerations:**
- 

---
## Summary and Submission

### Deliverables Checklist

- [ ] Task 1: Rubric definitions and pass/fail formula
- [ ] Task 2: Code for description generation
- [ ] Task 3: `assignment_01.xlsx` with manual evaluations (10-15 products)
- [ ] Task 4: Experiment documentation + code for successful experiments
- [ ] Task 5: Judge model implementation with Pydantic schema
- [ ] Task 6: Judge analysis, comparisons, and reflection

### Files to Submit

1. `assignment_01_solution.ipynb` (this notebook)
2. `assignment_01.xlsx` (with all evaluations)

**Due Date:** April 5, 2026